In [1]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PromptTuningConfig, PromptTuningInit, TaskType, get_peft_model

### Load SAMSum dataset

In [2]:
samsum = load_dataset("knkarthick/samsum")
print(samsum)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


_Make smaller splits_

In [3]:
train_raw = samsum["train"].shuffle(seed=42).select(range(3000))
val_raw = samsum["validation"].select(range(300))
test_raw = samsum["test"].select(range(300))

print("Train:", len(train_raw))
print("Val  :", len(val_raw))
print("Test :", len(test_raw))

Train: 3000
Val  : 300
Test : 300


_Show one sample_

In [4]:
def build_source_text(dialogue):
    return "Summarize this dialogue:\n" + dialogue.strip()

sample = train_raw[0]

print("Dialogue:\n")
print(sample["dialogue"])
print("\nReal Summary:\n")
print(sample["summary"])
print("\nDiscrete Prompt Text:\n")
print(build_source_text(sample["dialogue"]))

Dialogue:

Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface and installed it :D
Adam: yeah xd
Adam: ok i'll try to figure it out later
Hector: ok
Hector: i'll be waiting :P

Real Summary:

Adam will record it somewhere else through the interface and software. Hector gave and installed the interface before.

Discrete Prompt Text:

Summarize this dialogue:
Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface an